# Transformer(7. 2차 ResidualConnection_LayerNormalization) 언어 모델 총정리 

## 1. Decoder Block
	◦ 트랜스포머 디코더 블록 단계별 공식
	1) 입력값(token) 
	2) 임베딩 + 위치인코딩 
	3) Masked Multi-Head Attention 
	4) 1차 Residual Connection → 1차 Layer Normalization
	5) Cross Attention(인코더 Key/Value, 디코더 Query 사용하여 Attention 계산) 
	6) 2차 Residual Connection → 2차 Layer Normalization 
	7) Feed Forward Network(FFN)
	8) 3차 Residual Connection → 3차 Layer Normalization

---
### 1) 임베딩 + 위치 인코딩
$ [ Y = \text{Embedding}(tokens) + \text{PositionalEncoding} ] $

### 2) Masked Multi-Head Attention
$ [ \text{MaskedMHA}(Y) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V ] $

### 3) Residual Connection + Layer Normalization (1차)
$ [ H_1 = \text{LayerNorm}(Y + \text{MaskedMHA}(Y)) ] $

### 4) Cross Attention (인코더 Key/Value, 디코더 Query 사용하여 Attention 계산)
$ [ \text{CrossAttn}(H_1, H_{enc}) = \text{Softmax}\left(\frac{Q_{dec}K_{enc}^\top}{\sqrt{d_k}}\right)V_{enc} ] $

### 5) Residual Connection + Normalization (2차)
$ [ H_2 = \text{LayerNorm}(H_1 + \text{CrossAttn}(H_1, H_{enc})) ] $

### 6) Feed Forward Network (FFN)
$ [ \text{FFN}(H_2) = \max(0, H_2 W_1 + b_1) W_2 + b_2 ] $

### 7) Residual Connection + LayerNorm (3차)
$ [ H_3 = \text{LayerNorm}(H_2 + \text{FFN}(H_2)) ] $

---

## 2. Transformer(7. 2차 ResidualConnection_LayerNormalization) 언어 모델의 2차 ResidualConnection_LayerNormalization 계산 과정을 공식과 함께 단계별로 계산 
---
### 1) 2차 Residual Connection
	◦ 공식 :
$ [ X^{\prime\prime} = H_{dec} + \text{CrossAttention}(Q,K,V) ] $

- 디코더 입력 (1차 Residual + LayerNorm 결과):
$ [ H_{dec} = [-1.0, 1.0] ] $

- Cross-Attention 출력:
$ [ \text{CrossAttention}(Q,K,V) = [0,0] ] $

계산 : 

$ [ X^{\prime\prime} = [-1.0, 1.0] + [0,0] = [-1.0, 1.0] ] $

참고) Residual Connection 덕분에 입력값이 그대로 유지됩니다.

---
### 2) 2차 Layer Normalization
	◦ 공식 :
$ [ \text{LayerNorm}(X^{\prime\prime}) = \frac{X^{\prime\prime} - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta ] $

2-1) 평균 :
$ [ \mu = \frac{-1.0 + 1.0}{2} = 0 ] $

2-2) 분산 :
$ [ \sigma^2 = \frac{(-1.0-0)^2 + (1.0-0)^2}{2} = \frac{1+1}{2} = 1 ] $

2-3) 표준편차 :
$ [ \sqrt{\sigma^2 + \epsilon} \approx 1 ] $

2-4) 정규화 :
$ [ \frac{[-1.0, 1.0] - 0}{1} = [-1.0, 1.0] ] $

참고) 현재는 필요없음, 여기에 학습 파라미터 [ \gamma, \beta ]가 적용됩니다.

예를 들어 [ \gamma = 1, \beta = 0 ]이라면 결과는 그대로 [ [-1.0, 1.0] ]입니다.

---